In [ ]:
import numpy as np
import pandas as pd
from keras.models import Sequential
from keras.layers import Conv1D, MaxPooling1D, LSTM, Dense, Flatten, Dropout
from sklearn.preprocessing import MinMaxScaler

# Load and preprocess your stock market data (e.g., historical closing prices)
data = pd.read_csv('stock_data.csv')  # Example CSV file
prices = data['Close'].values

# Scale the data to the range [0, 1]
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_prices = scaler.fit_transform(prices.reshape(-1, 1))

# Define a function to create the data structure for the CNN-LSTM model
def create_dataset(data, time_step=60):
    X, y = [], []
    for i in range(len(data) - time_step - 1):
        X.append(data[i:(i + time_step), 0])  # Sliding window of 60 days
        y.append(data[i + time_step, 0])  # Next day's stock price
    return np.array(X), np.array(y)

# Prepare the data
time_step = 60
X, y = create_dataset(scaled_prices, time_step)

# Reshape the input data for CNN input (samples, time steps, features)
X = X.reshape(X.shape[0], X.shape[1], 1)

# Split the data into training and testing sets
train_size = int(len(X) * 0.8)
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

# Build the CNN-LSTM model
model = Sequential()

# CNN Layer
model.add(Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=(X_train.shape[1], 1)))
model.add(MaxPooling1D(pool_size=2))

# LSTM Layer
model.add(LSTM(units=50, return_sequences=True))
model.add(Dropout(0.2))

# Flatten and Dense Layers
model.add(Flatten())
model.add(Dense(units=1))

# Compile the model
model.compile(optimizer='adam', loss='mean_squared_error')

# Train the model
model.fit(X_train, y_train, epochs=10, batch_size=32)

# Evaluate the model
predictions = model.predict(X_test)

# Inverse scaling to get the real stock prices
predictions = scaler.inverse_transform(predictions)
y_test = scaler.inverse_transform(y_test.reshape(-1, 1))

# Evaluate model performance
print(f"Predicted stock prices: {predictions[:5]}")
print(f"Actual stock prices: {y_test[:5]}")
